# Phase 4: Machine Learning Risk Scoring Model

## Project: Advanced Health Analytics & Readmission Prediction

This is the final stage of our clinical data pipeline. We use the insights gained from exploratory and statistical analysis to build a predictive model for 30-day hospital readmission.

### Objectives:
1. **Feature Engineering**: Preparing clinical and demographic data for ML.
2. **Model Training**: Comparing Logistic Regression (interpretability) and Random Forest (performance).
3. **Performance Evaluation**: Using ROC-AUC and Confusion Matrices.
4. **Deployment Simulation**: Creating a 'Risk Scoring' function for clinical use.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [10, 6]

### 1. Data Loading & Preprocessing
We load the raw data and apply the same cleaning steps established in Phase 1.

In [ ]:
data_path = "../data/raw/synthetic_clinical_data.csv"
df = pd.read_csv(data_path)

# Pre-processing pipeline
df_clean = df.dropna(subset=['Admission_Date']).copy()
df_clean['Hemoglobin_gdL'] = df_clean['Hemoglobin_gdL'].fillna(df_clean['Hemoglobin_gdL'].median())
df_clean['Systolic_BP'] = df_clean['Systolic_BP'].clip(60, 220)
df_clean['Glucose_mgdL'] = df_clean['Glucose_mgdL'].clip(40, 400)

# Encode Gender
le = LabelEncoder()
df_clean['Gender_Encoded'] = le.fit_transform(df_clean['Gender'])

print(f"Preprocessing complete. Shape: {df_clean.shape}")

### 2. Feature Selection & Train/Test Split
We select clinical markers identified as significant in Phase 2.

In [ ]:
features = ['Age', 'Gender_Encoded', 'Stay_Duration_Days', 'Systolic_BP', 'Glucose_mgdL', 'Diabetes', 'Hypertension']
X = df_clean[features]
y = df_clean['Readmitted_30Days']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Standardize values
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data split into 80% Training and 20% Testing sets.")

### 3. Model Implementation: Random Forest
We train a Random Forest classifier to identify high-risk patients.

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

y_pred = rf_model.predict(X_test_scaled)
y_prob = rf_model.predict_proba(X_test_scaled)[:, 1]

print("### Random Forest Performance")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")

### 4. Clinical Deployment Tool: Risk Scoring Function
A reusable function to predict risk levels for new patient intake.

In [ ]:
def predict_readmission_risk(patient_data):
    """
    Input: dict with keys matching features
    Output: Risk level string
    """
    patient_df = pd.DataFrame([patient_data])
    patient_scaled = scaler.transform(patient_df)
    prob = rf_model.predict_proba(patient_scaled)[0, 1]
    
    if prob > 0.7: return "HIGH RISK"
    if prob > 0.4: return "MEDIUM RISK"
    return "LOW RISK"

# Example usage
sample_patient = {
    'Age': 75,
    'Gender_Encoded': 0, # F
    'Stay_Duration_Days': 10,
    'Systolic_BP': 150,
    'Glucose_mgdL': 250,
    'Diabetes': 1,
    'Hypertension': 1
}

risk = predict_readmission_risk(sample_patient)
print(f"Predicted risk for sample patient: {risk}")